# GBFS Audit Catalogue — Recipes

Companion notebook to the **GBFS Audit Catalogue** v1.0 (Fossé & Pallares, 2026).

- **DOI**: [10.5281/zenodo.20125460](https://doi.org/10.5281/zenodo.20125460)
- **Licence**: ODbL v1.0 for the data; MIT for the code.
- **Citation**: please cite the Computer Standards & Interfaces paper plus the Zenodo deposit DOI when reusing the catalogue.

This notebook reproduces and extends the five use cases of the paper. Each recipe is self-contained: load the parquet, apply the snippet, look at the result. All examples are typed against the 46-column schema documented in the manuscript (Table 7).

## What you'll find here

| # | Use case | Columns featured |
|---|---|---|
| 1 | Loading the catalogue | `station_type`, `audit_confidence` |
| 2 | Per-operator anomaly profile | `operator_name`, `flag_A1`–`flag_A7` |
| 3 | Raw vs audited capacity per system | `capacity_raw`, `capacity_audited` |
| 4 | Spatial-equity correlation | `revenu_median_uc`, `station_type` |
| 5 | Identifying mobility deserts | `gtfs_heavy_stops_300m`, `revenu_median_uc` |
| 6 | (bonus) Per-region station mass | `region_id`, `station_type` |
| 7 | (bonus) Audit confidence breakdown | `audit_confidence` |
| 8 | (bonus) Network density per system | `dist_to_nearest_station_m`, `n_stations_within_500m` |

## Setup

The catalogue is a single ~6.6 MB Parquet file. Three ways to load it:

- **Local** (after downloading from Zenodo): `pd.read_parquet("stations_gold_standard_final.parquet")`
- **Direct from Zenodo** (no auth, one HTTP fetch):
  ```python
  pd.read_parquet("https://zenodo.org/records/20125460/files/stations_gold_standard_final.parquet")
  ```
- **Via Hugging Face Datasets** (with caching):
  ```python
  from datasets import load_dataset
  gs = load_dataset("rohanfosse/gbfs-audit-catalogue", split="train").to_pandas()
  ```

The cell below uses the local path (works when you run the notebook from the repository root).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

# Local path; replace with the Zenodo URL for remote access.
PARQUET = Path("../../../data/stations_gold_standard_final.parquet")
if not PARQUET.exists():
    PARQUET = Path("data/stations_gold_standard_final.parquet")

gs = pd.read_parquet(PARQUET)
print(f"Loaded {len(gs):,} stations x {len(gs.columns)} columns from {PARQUET}")
print(f"Schema preview:")
print(gs.dtypes.head(15))

Loaded 46,307 stations x 46 columns from data\stations_gold_standard_final.parquet
Schema preview:
uid                               object
system_id                         object
city                              object
system_name                       object
source                            object
station_id                        object
station_name                      object
lat                              float64
lon                              float64
capacity                         float64
address                           object
region_id                         object
n_stations_system                float64
fetched_at           datetime64[ns, UTC]
elevation_m                      float64
dtype: object


## Use case 1 — Loading the catalogue (high-confidence subset)

Most analyses want the subset of stations that the audit certifies *without a flag*. The `audit_confidence` column captures this in a single ordinal value (`high`, `medium`, `low`). Combine it with `station_type` to get the canonical, fully-audited dock-based stations.

In [2]:
clean = gs[(gs.station_type == "docked_bike") & (gs.audit_confidence == "high")]
print(f"High-confidence dock-based stations: {len(clean):,}")
print(f"\nDistribution of audit_confidence on the full corpus:")
print(gs.audit_confidence.value_counts())

High-confidence dock-based stations: 5,402

Distribution of audit_confidence on the full corpus:
audit_confidence
low       34360
medium     6545
high       5402
Name: count, dtype: int64


## Use case 2 — Per-operator anomaly profile

The seven flag columns make the audit's verdict inspectable at the row level. Aggregating them by `operator_name` recovers the operator-driven hotspots that the paper documents (Dott → A7, Pony → A3).

*Reading the table*: `A7_rate = 1.0` means every station of that operator has its capacity field declared as NaN; `A3_rate = 1.0` means every station is published as a free-floating anchor.

In [3]:
op = (
    gs.groupby("operator_name")
      .agg(
          n=("uid", "size"),
          A1_rate=("flag_A1", "mean"),
          A2_rate=("flag_A2", "mean"),
          A3_rate=("flag_A3", "mean"),
          A6_rate=("flag_A6", "mean"),
          A7_rate=("flag_A7", "mean"),
      )
      .sort_values("n", ascending=False)
      .head(10)
)
op

,n,A1_rate,A2_rate,A3_rate,A6_rate,A7_rate
operator_name,,,,,,
Dott,20224,0.0,0.000000,1.0,0.0,0.999604
Pony,11465,0.0,0.035935,1.0,0.0,0.648583
Bird,4499,0.0,0.000000,1.0,0.0,1.000000
Voi,2816,0.0,0.000000,1.0,0.0,0.000000
Citiz (carsharing),1547,1.0,0.000000,0.0,0.0,0.000000
Vélib' Métropole,1533,0.0,0.000000,0.0,0.0,0.000000
Vélo'V,452,0.0,0.000000,0.0,0.0,0.000000
VélÔToulouse,434,0.0,0.000000,0.0,0.0,0.000000
V'lille,268,0.0,0.000000,0.0,0.0,0.000000


## Use case 3 — Raw vs audited capacity per system

`capacity_raw` preserves the GBFS-declared value (placeholders, NaN and all); `capacity_audited` carries the post-audit value (NaN for non-dock types). The ratio of total raw capacity to total audited capacity quantifies the **over-counting factor** of a naive GBFS pipeline at the system level.

The table below highlights the systems where the raw feed claims tens of thousands of docks that the audit reduces to almost nothing.

In [4]:
sys = (
    gs.groupby("system_id")
      .agg(
          n_stations=("uid", "size"),
          operator=("operator_name", "first"),
          raw_capacity=("capacity_raw", "sum"),
          audited_capacity=("capacity_audited", "sum"),
      )
      .query("raw_capacity > 1000")
)
sys["over_count_factor"] = sys.raw_capacity / sys.audited_capacity.replace(0, 1)
sys.sort_values("over_count_factor", ascending=False).head(10)

,n_stations,operator,raw_capacity,audited_capacity,over_count_factor
system_id,,,,,
pony_Nice,412,Pony,41200.0,0.0,41200.0
voi_Grenoble,697,Voi,6782.0,0.0,6782.0
voi_Marseille,1267,Voi,6618.0,0.0,6618.0
pony_paris,3617,Pony,5885.0,0.0,5885.0
voi_Le_Havre,470,Voi,4288.0,0.0,4288.0
voi_SQY,382,Voi,3067.0,0.0,3067.0
citiz_alpes_loire,408,Citiz (carsharing),1060.0,0.0,1060.0
Paris,1507,Vélib' Métropole,48410.0,48410.0,1.0
CVelo_FR_Clermont-Ferrand,82,C-Vélo,1716.0,1716.0,1.0


## Use case 4 — Spatial-equity correlation

Per-city correlation between dock-based station count and median income per consumption unit (INSEE Filosofi). A positive correlation indicates that wealthier cities tend to have larger dock-based networks.

In [5]:
by_city = (
    gs[gs.station_type == "docked_bike"]
      .groupby("city")
      .agg(n=("uid", "size"), income=("revenu_median_uc", "median"))
      .dropna()
)
rho = by_city.n.corr(by_city.income, method="spearman")
print(f"Spearman rho(n_stations, median_income) = {rho:+.3f}  (n_cities = {len(by_city)})")
print("\nTop 10 cities by station count:")
by_city.sort_values("n", ascending=False).head(10)

Spearman rho(n_stations, median_income) = +0.094  (n_cities = 64)

Top 10 cities by station count:


,n,income
city,,
Paris,1507,30760.0
Lyon,452,23390.0
Toulouse,434,21040.0
Lille,268,18620.0
Bordeaux,225,23710.0
Marseille,221,21870.0
Rouen,131,20150.0
Nantes,124,23020.0
Brest,115,19870.0


## Use case 5 — Identifying mobility deserts

A *cycling-equity desert* is a dock-based station that sits simultaneously in a commune in the lowest income quartile **and** without any heavy-transit stop within 300 m. The filter below recovers ≈1,041 stations (19.1% of the dock-based corpus) and ranks the contributing French urban areas.

In [6]:
q1_income = gs.revenu_median_uc.quantile(0.25)
deserts = gs[
    (gs.station_type == "docked_bike")
    & (gs.revenu_median_uc < q1_income)
    & (gs.gtfs_heavy_stops_300m == 0)
]
print(f"Income Q1 threshold = {q1_income:,.0f} EUR per consumption unit")
print(f"Mobility-desert stations: {len(deserts):,} ({100 * len(deserts) / (gs.station_type == 'docked_bike').sum():.1f}% of dock-based)")
print("\nTop 10 contributing urban areas:")
deserts.city.value_counts().head(10)

Income Q1 threshold = 21,040 EUR per consumption unit
Mobility-desert stations: 1,682 (30.9% of dock-based)

Top 10 contributing urban areas:


city
Lille               205
Lyon                174
Paris               159
Marseille            99
Rouen                82
Épinal               73
Clermont-Ferrand     56
Saint-Étienne        56
Valence              55
Calais               52
Name: count, dtype: int64

## Use case 6 — (bonus) Per-region station mass

Aggregating by administrative region surfaces the geographic concentration patterns of the certified network. Île-de-France dominates the dock-based mass (Vélib' Métropole alone), while regions with large free-floating fleets contribute much smaller dock-based stocks.

In [7]:
dock = gs[gs.station_type == "docked_bike"]
region = (
    dock.groupby("region_id")
        .agg(
            n_stations=("uid", "size"),
            n_systems=("system_id", "nunique"),
            mean_capacity=("capacity_audited", "mean"),
        )
        .sort_values("n_stations", ascending=False)
)
region

,n_stations,n_systems,mean_capacity
region_id,,,
689,75,1,5.373333
region_rennes_metropole_35238,57,1,24.736842
1124,48,1,16.583333
494,40,1,4.725
934,25,1,15.48
1163,22,1,16.318182
1162,7,1,17.142857
1157,5,1,15.0
1158,3,1,17.666667


## Use case 7 — (bonus) Audit confidence breakdown by station type

Cross-tabulating `station_type` with `audit_confidence` shows where the high-confidence mass sits. Only dock-based stations reach the `high` confidence tier; free-floating entries are systematically tagged `low` because of the A3/A7 flag co-occurrence.

In [8]:
pd.crosstab(
    gs.station_type,
    gs.audit_confidence,
    margins=True,
    margins_name="total",
)

audit_confidence,high,low,medium,total
station_type,,,,
carsharing,0,1630,0,1630
docked_bike,5402,40,0,5442
free_floating,0,32690,6545,39235
total,5402,34360,6545,46307


## Use case 8 — (bonus) Network density per system

The Tier 2 columns `dist_to_nearest_station_m`, `n_stations_within_500m` and `catchment_density_per_km2` expose intra-system geometry without recomputing the KNN. The histogram of mean nearest-neighbour distance by system reveals the typology of French BSS networks: very dense centre-city Vélib'-style versus diffuse free-floating deployments.

In [9]:
geom = (
    gs[gs.station_type == "docked_bike"]
      .groupby("system_id")
      .agg(
          n=("uid", "size"),
          city=("city", "first"),
          mean_nn_m=("dist_to_nearest_station_m", "mean"),
          median_density_500m=("n_stations_within_500m", "median"),
      )
      .query("n >= 30")
      .sort_values("mean_nn_m")
)
geom.head(15)

,n,city,mean_nn_m,median_density_500m
system_id,,,,
donkey_valenciennes,40,Valenciennes,139.225315,8.0
donkey_brest,75,Brest,181.987975,11.0
Paris,1507,Paris,290.436436,6.0
nantes,124,Nantes,300.211396,4.0
inurba-rouen,131,Rouen,311.598141,3.0
levelo_inurba_marseille,221,Marseille,338.454118,3.0
toulouse,434,Toulouse,352.326658,3.0
v_lille,268,Lille,370.007344,2.0
CVelo_FR_Clermont-Ferrand,82,Clermont-Ferrand,371.807409,2.0


## Citation

If you use the GBFS Audit Catalogue in your research, please cite both the paper and the Zenodo deposit:

> Fossé, R. & Pallares, G. (2026). *Auditing GBFS bike-sharing feeds at country and global scale: A reproducible anomaly taxonomy for open mobility data*. Computer Standards & Interfaces. DOI: TBD.

> Fossé, R. & Pallares, G. (2026). *GBFS Audit Catalogue v1.0* [Data set]. Zenodo. https://doi.org/10.5281/zenodo.20125460

Issues and contributions are welcome at <https://github.com/rohanfosse/bikeshare-data-explorer>.